# Analysis of DRL Resource Allocation Experiments

This notebook is used to analyze the results of the DRL-based 5G resource allocation experiments. It loads the data from the `results` directory, processes it, and generates plots and tables for the paper.

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="whitegrid")

# Define paths
results_dir = Path("../results/openran_live_prom_randomized_100ue_true_10seed")
output_fig_dir = Path("paper_figures")
output_fig_dir.mkdir(exist_ok=True)

# Function to load and process summary data
def load_summary(results_dir: Path):
    summary_files = results_dir.glob("**/summary.csv")
    all_summaries = []
    for f in summary_files:
        algo_type = f.parent.name
        df = pd.read_csv(f)
        df['algorithm'] = algo_type
        all_summaries.append(df)
    return pd.concat(all_summaries, ignore_index=True)

# Function to plot evolution of metrics
def plot_metric_evolution(results_dir: Path, metric: str, title: str, filename: str):
    run_logs = results_dir.glob("**/run_log.csv")
    plt.figure(figsize=(12, 7))
    
    for log_file in run_logs:
        df = pd.read_csv(log_file)
        algo_type = log_file.parent.parent.name
        seed = log_file.parent.name
        if "train" in algo_type:
            plt.plot(df.index, df[metric], label=f"{algo_type} (seed {seed})")

    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.savefig(output_fig_dir / filename)
    plt.show()

# Load summary data
summary_df = load_summary(results_dir)
print("Summary Data:")
print(summary_df)

# Generate and save summary table for the paper
summary_table = summary_df.groupby('algorithm').agg({
    'total_reward': ['mean', 'std'],
    'avg_total_violation': ['mean', 'std']
}).reset_index()
summary_table.columns = ['Algorithm', 'Mean Reward', 'Reward Std', 'Mean SLA Violation', 'SLA Violation Std']
print("\nPaper Summary Table:")
print(summary_table.to_latex(index=False, float_format="%.3f"))

# Plot and save metric evolution
plot_metric_evolution(results_dir, "total_reward", "Evolution of Total Reward During Training", "reward_evolution.png")
plot_metric_evolution(results_dir, "avg_total_violation", "Evolution of SLA Violations During Training", "violation_evolution.png")

# Bar charts for comparison are already generated by the scripts, but we can recreate them here for consistency
plt.figure(figsize=(10, 6))
sns.barplot(data=summary_df, x='algorithm', y='total_reward', capsize=.1)
plt.title('Comparison of Total Reward')
plt.savefig(output_fig_dir / "reward_comparison_notebook.png")
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=summary_df, x='algorithm', y='avg_total_violation', capsize=.1)
plt.title('Comparison of Average SLA Violation')
plt.savefig(output_fig_dir / "violation_comparison_notebook.png")
plt.show()


# DRL Slicing Experiment Analysis

Load and visualize multi-seed experiment outputs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load summary if available
summary_path = Path('results/summary.csv')
if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    print('Summary statistics across seeds:')
    print(df_summary.describe())
else:
    print('No summary.csv found. Run batch_runner.py first.')

## Load a single run

In [ ]:
# Load the test run log
test_log = Path('results/test_run_log.csv')
if test_log.exists():
    df_run = pd.read_csv(test_log)
    print(f'Loaded {len(df_run)} steps')
    print('\nFirst 5 rows:')
    print(df_run.head())
    print('\nReward statistics:')
    print(df_run['reward'].describe())
    print(f'\nTotal SLA violations: {df_run["sla_violations"].sum()}')